# Import libraries

In [1]:
import os
import shutil
import pandas as pd
import subprocess
from tqdm import tqdm
import concurrent.futures

import sys
sys.path.append('..')
from utils.audio_util import validate_wav_files, convert_wav_to_flac, resample_audios, trim_silence_with_vad, normalize_audio_files
from utils.file_util import recursive_copy

/home/pruuwu/dubbing-ai/Restructure/converter/../utils/audio_util.py:12: UserWarning: Module 'speechbrain.pretrained' was deprecated, redirecting to 'speechbrain.inference'. Please update your script. This is a change from SpeechBrain 1.0. See: https://github.com/speechbrain/speechbrain/releases/tag/v1.0.0
  from speechbrain.pretrained import SepformerSeparation as separator


# Check invalid wav file

In [2]:
# Make sure wav is valid
validate_wav_files("../data/raw/VCTK-Corpus/wav48")

Found 44242 WAV files to validate


Validating WAV files: 100%|██████████| 44242/44242 [30:23<00:00, 24.27it/s]

Successfully validated 44242 WAV files


44242

# Move file

In [8]:
SRC_PATH = "../data/raw/VCTK-Corpus"
DEST_PATH = "../data/converted/vctk-to-vctk"

SRC_TEXT_PATH = os.path.join(SRC_PATH, 'txt')
SRC_WAV_PATH = os.path.join(SRC_PATH, 'wav48')
DEST_TEXT_PATH = os.path.join(DEST_PATH, 'txt')
DEST_WAV_PATH = os.path.join(DEST_PATH, 'wav48')

# Create destination directories if they don't exist
if not os.path.exists(DEST_WAV_PATH):
    os.makedirs(DEST_WAV_PATH)
if not os.path.exists(DEST_TEXT_PATH):
    os.makedirs(DEST_TEXT_PATH)

In [9]:
def process_file(item):
    """Process a single WAV file from VCTK dataset with proper directory structure."""
    speaker_id, wav_file, count = item
    
    src_wav_file = os.path.join(SRC_WAV_PATH, speaker_id, wav_file)
    base_filename = os.path.splitext(wav_file)[0]
    
    # Create speaker directories if they don't exist - use exist_ok=True to avoid race conditions
    dest_speaker_wav_dir = os.path.join(DEST_WAV_PATH, speaker_id)
    dest_speaker_txt_dir = os.path.join(DEST_TEXT_PATH, speaker_id)
    
    os.makedirs(dest_speaker_wav_dir, exist_ok=True)
    os.makedirs(dest_speaker_txt_dir, exist_ok=True)
    
    # Define source and destination paths
    src_txt_file = os.path.join(SRC_TEXT_PATH, speaker_id, f"{base_filename}.txt")
    dest_wav_file = os.path.join(dest_speaker_wav_dir, f"{base_filename}_mic1.flac")
    dest_txt_file = os.path.join(dest_speaker_txt_dir, f"{base_filename}.txt")
    
    try:
        # Copy text file from source to destination
        if os.path.exists(src_txt_file):
            shutil.copy(src_txt_file, dest_txt_file)
        else:
            print(f"Warning: Text file not found at {src_txt_file}")
        
        # Convert audio file
        convert_wav_to_flac(src_wav_file, dest_wav_file)
        
        return True
    except Exception as e:
        print(f"Error processing {speaker_id}/{wav_file}: {str(e)}")
        return False

# Check if source directory exists
if not os.path.exists(SRC_WAV_PATH):
    print(f"Error: Directory not found at {SRC_WAV_PATH}")
else:
    # Get all speaker directories
    speaker_dirs = [d for d in os.listdir(SRC_WAV_PATH) if os.path.isdir(os.path.join(SRC_WAV_PATH, d))]
    
    # Prepare the items to process
    items_to_process = []
    count = 1
    
    for speaker_id in speaker_dirs:
        speaker_wav_dir = os.path.join(SRC_WAV_PATH, speaker_id)
        wav_files = [f for f in os.listdir(speaker_wav_dir) if f.endswith('.wav')]
        
        for wav_file in wav_files:
            items_to_process.append((speaker_id, wav_file, count))
            count += 1
    
    # Use ThreadPoolExecutor for parallel processing
    with concurrent.futures.ThreadPoolExecutor(max_workers=os.cpu_count()) as executor:
        # Use tqdm to show progress
        results = list(tqdm(
            executor.map(process_file, items_to_process),
            total=len(items_to_process),
            desc="Processing VCTK audio files"
        ))
    
    # Report results
    successful = results.count(True)
    failed = results.count(False)
    print(f"Processing complete: {successful} files processed successfully, {failed} files failed")

Processing VCTK audio files:  91%|█████████▏| 40380/44242 [03:57<00:23, 163.01it/s]

Processing VCTK audio files:  91%|█████████▏| 40415/44242 [03:57<00:23, 164.01it/s]

Processing VCTK audio files:  91%|█████████▏| 40435/44242 [03:58<00:22, 172.22it/s]

Processing VCTK audio files:  91%|█████████▏| 40470/44242 [03:58<00:23, 162.95it/s]

Processing VCTK audio files:  92%|█████████▏| 40508/44242 [03:58<00:21, 170.16it/s]

Processing VCTK audio files: 100%|██████████| 44242/44242 [04:20<00:00, 169.66it/s]

Processing complete: 44242 files processed successfully, 0 files failed


Note: p315 does not have text

# Resample, trim, and normalize audio

In [10]:
# Create destination directory if it doesn't exist
os.makedirs("../data/converted/vctk-to-vctk/wav16_silence_trimmed", exist_ok=True)

# Copy all files from wav44 to wav16_silence_trimmed
src_dir = "../data/converted/vctk-to-vctk/wav48"
dst_dir = "../data/converted/vctk-to-vctk/wav16_silence_trimmed"

recursive_copy(src_dir, dst_dir)

In [11]:
# Resample all files in wav16_silence_trimmed to 16kHz
SAMPLE_RATE = 16000
NUM_RESAMPLE_THREADS = 8

resample_audios(
  input_folders=dst_dir,
  file_ext="flac",
  sample_rate=SAMPLE_RATE,
  n_jobs=NUM_RESAMPLE_THREADS
)

Resampling the audio files...
Found 44242 files...


100%|██████████| 44242/44242 [00:40<00:00, 1088.18it/s]

Done !


In [12]:
# Trim silence at the beginning and end of each audio file
trim_silence_with_vad(
  input_folder=dst_dir,
  file_extension="flac",
)

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /home/pruuwu/.cache/torch/hub/master.zip


Found 44242 .flac files to process


Processing files:  21%|██        | 9254/44242 [08:39<27:23, 21.29it/s]  

> The file ../data/converted/vctk-to-vctk/wav16_silence_trimmed/p323/p323_424_mic1.flac probably does not have speech please check it !!


Processing files:  38%|███▊      | 16863/44242 [16:05<24:34, 18.57it/s]

...

> The file ../data/converted/vctk-to-vctk/wav16_silence_trimmed/p341/p341_101_mic1.flac probably does not have speech please check it !!


Processing files:  97%|█████████▋| 43129/44242 [41:22<01:15, 14.65it/s]

> The file ../data/converted/vctk-to-vctk/wav16_silence_trimmed/p300/p300_155_mic1.flac probably does not have speech please check it !!


Processing files: 100%|██████████| 44242/44242 [42:28<00:00, 17.36it/s]


Processing complete

Found 8 files with no speech. List saved to ../data/converted/vctk-to-vctk/no_speech_files.txt


In [13]:
# Normalize the volume of all audio files to -27dB
normalize_audio_files(
    input_dir=dst_dir,
)

Normalizing audio files: 100%|██████████| 44242/44242 [2:28:59<00:00,  4.95it/s]  


# Output stat

In [14]:
import os
import glob
import soundfile as sf
from tqdm import tqdm

# Path pattern
path_pattern = "../data/converted/vctk-to-vctk/wav16_silence_trimmed/**/*.flac"

# Get all flac files matching the pattern
flac_files = glob.glob(path_pattern, recursive=True)

if not flac_files:
    print(f"No FLAC files found matching pattern: {path_pattern}")
else:
    print(f"Found {len(flac_files)} FLAC files. Processing...")
    
    # Calculate total duration
    total_duration_seconds = 0
    
    # Track speaker folders
    speaker_folders = set()
    
    # Extract base directory for later use in calculating speaker directories
    base_dir = os.path.normpath("../data/converted/vctk-to-vctk/wav16_silence_trimmed")
    
    # Use tqdm for progress bar
    for flac_file in tqdm(flac_files):
        try:
            # Get audio info
            info = sf.info(flac_file)
            total_duration_seconds += info.duration
            
            # Extract speaker folder - take the directory right after wav16_silence_trimmed/
            rel_path = os.path.relpath(os.path.dirname(flac_file), base_dir)
            if '/' in rel_path:
                speaker = rel_path.split('/')[0]  # First directory is the speaker
            else:
                speaker = rel_path  # If there's no further nesting
                
            speaker_folders.add(speaker)
            
        except Exception as e:
            print(f"Error processing {flac_file}: {e}")
    
    # Convert to hours
    total_duration_hours = total_duration_seconds / 3600
    
    # Print results
    print(f"\nTotal duration: {total_duration_hours:.2f} hours")
    print(f"                ({total_duration_hours*60:.2f} minutes)")
    print(f"                ({total_duration_hours*3600:.2f} seconds)")
    
    # Print only the number of speakers
    print(f"Number of speakers: {len(speaker_folders)}")

Found 44242 FLAC files. Processing...


100%|██████████| 44242/44242 [00:03<00:00, 14724.66it/s]


Total duration: 25.37 hours
                (1521.93 minutes)
                (91315.68 seconds)
Number of speakers: 109
